# FlashRank-Pro Training Pipeline
---
**Built by [Eulogik](https://eulogik.com)**

Resumable notebook for Colab T4. Each stage saves to Google Drive.
If the session drops, just **Runtime → Run all** — it picks up where it left off.

---
## Before you start

### Get this notebook into Colab
Since the repo is private, use one of these:
  - **A)** Download notebook from your local clone and upload to Colab (File → Upload notebook)
  - **B)** Add `GH_TOKEN` secret (🔑 Secrets panel) with a GitHub PAT scoped to `repo` — then Colab can clone directly

### Set secrets (optional)
In the 🔑 **Secrets** panel (left sidebar):
| Secret | Required for | How to get |
|--------|-------------|-----------|
| `GH_TOKEN` | Cloning private repo in Colab | GitHub Settings → Developer settings → Personal access tokens → `repo` scope |
| `LLM_API_KEY` | Stage 1 — optional, for OpenRouter/OpenAI query gen | Provider's API key page |
| `HF_TOKEN` | Deploy to HuggingFace | [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) |

### Enable GPU
Runtime → Change runtime type → **T4 GPU**

---

In [ ]:
# ============================================================
# SETUP: Install deps, mount Drive, get repo, resume progress
# ============================================================
import os, sys, json, shutil, subprocess, time, warnings
from pathlib import Path

DRIVE_MOUNT = "/content/drive"
DRIVE_ROOT  = f"{DRIVE_MOUNT}/MyDrive/flashrank-pro"
LOCAL_ROOT  = "/content/flashrank-pro"
GIT_REPO    = "https://github.com/eulogik/flashrank-pro.git"

print("⚡ FlashRank-Pro Training Pipeline")
print("=" * 50)

# ---- 1. Install dependencies (idempotent, Colab caches pip) ----
print("\n[1/6] Installing dependencies...")
start = time.time()
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q transformers sentence-transformers accelerate datasets fire tqdm peft openai huggingface-hub
print(f"    ✅ Dependencies ready ({time.time()-start:.0f}s)")

# ---- 2. Mount Google Drive ----
print("\n[2/6] Mounting Google Drive...")
from google.colab import drive
drive.mount(DRIVE_MOUNT, force_remount=False)
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"    ✅ Drive mounted")

# ---- 3. Get the repo code ----
print("\n[3/6] Getting training scripts...")

def clone_repo_with_token():
    """Clone private repo using GH_TOKEN from Secrets."""
    from google.colab import userdata
    token = userdata.get("GH_TOKEN")
    authed_url = f"https://{token}@github.com/eulogik/flashrank-pro.git"
    subprocess.run(["git", "clone", authed_url, LOCAL_ROOT], check=True, capture_output=True)
    return True

if os.path.exists(LOCAL_ROOT):
    print("    ✅ Code already present locally — pulling latest...")
    subprocess.run(["git", "-C", LOCAL_ROOT, "pull"], check=True, capture_output=True)
elif os.path.exists(f"{LOCAL_ROOT}/training/02_knowledge_distillation.py"):
    print("    ✅ Code already present locally — pulling latest...")
    subprocess.run(["git", "-C", LOCAL_ROOT, "pull"], check=True, capture_output=True)
else:
    cloned = False
    # Try GH_TOKEN from Secrets first
    try:
        from google.colab import userdata
        _ = userdata.get("GH_TOKEN")
        if _:
            print("    Found GH_TOKEN, cloning private repo...")
            authed_url = f"https://{_}@github.com/eulogik/flashrank-pro.git"
            subprocess.run(["git", "clone", authed_url, LOCAL_ROOT], check=True, capture_output=True)
            cloned = True
            print("    ✅ Cloned via GH_TOKEN")
    except Exception:
        pass

    # Fallback: try public clone (will fail for private, but user might have SSH)
    if not cloned:
        print("    No GH_TOKEN found. Trying public clone...")
        try:
            subprocess.run(["git", "clone", GIT_REPO, LOCAL_ROOT], check=True, capture_output=True, timeout=30)
            cloned = True
            print("    ✅ Cloned (public access or SSH key)")
        except Exception as e:
            print(f"    ❌ Public clone failed: {e}")

    # Last resort: ask user to upload the files
    if not cloned:
        print("\n" + "=" * 50)
        print("⚠️  Could not clone private repo.")
        print("")
        print("Options:")
        print("  1. Add GH_TOKEN secret: 🔑 Secrets panel → Add new secret → GH_TOKEN")
        print("     (GitHub PAT with `repo` scope)")
        print("  2. Run this cell anyway with the file upload below")
        print("")
        print("Upload the repo zip from your local machine:")
        from google.colab import files
        uploaded = files.upload()  # Upload flashrank-pro zip
        for fname in uploaded:
            !unzip -q {fname} -d /content/
            # Handle zip with root folder or without
            if os.path.exists(f"/content/{fname.replace('.zip','')}"):
                !mv /content/{fname.replace('.zip','')} {LOCAL_ROOT}
            print(f"    ✅ Extracted {fname}")

if not os.path.exists(LOCAL_ROOT):
    os.makedirs(LOCAL_ROOT, exist_ok=True)

os.chdir(LOCAL_ROOT)
sys.path.insert(0, LOCAL_ROOT)
print(f"    Working dir: {os.getcwd()}")

# ---- 4. Restore previous progress from Drive ----
print("\n[4/6] Restoring previous progress from Drive...")
for item in ["data", "models"]:
    local_p = Path(LOCAL_ROOT) / item
    drive_p = Path(DRIVE_ROOT) / item
    if drive_p.exists() and not local_p.exists():
        shutil.copytree(str(drive_p), str(local_p))
        print(f"    ✅ Restored {item}/ from Drive")
    elif local_p.exists():
        print(f"    ✅ {item}/ already present")
    else:
        local_p.mkdir(parents=True, exist_ok=True)
        print(f"    📁 Created {item}/")

# ---- 5. Check GPU ----
print("\n[5/6] Checking GPU...")
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"    ✅ {gpu} ({mem:.0f} GB VRAM)")
else:
    print("    ⚠️  No GPU — training will be slow")

# ---- 6. Summary ----
print("\n[6/6] Setup complete")
stages = [
    ("data",     "data/synthetic_training_data.jsonl"),
    ("kd_en",    "models/flashrank-pro-base-kd-en"),
    ("kd_multi", "models/flashrank-pro-base-kd-multilingual"),
    ("rl",       "models/flashrank-pro-base-rl"),
    ("merged",   "models/flashrank-pro-merged"),
]
for name, path in stages:
    done = os.path.exists(path)
    print(f"    {'✅' if done else '⬜'} {name}: {'done' if done else 'pending'}")
print("=" * 50)

---
## Stage 1: Generate training data
**Cost:** $0 (uses HuggingFace datasets) | **Runtime:** ~30min

Uses GooAQ's existing question→answer pairs. No API key needed.
Skip if you already have `data/synthetic_training_data.jsonl`.

In [ ]:
# ============================================================
# STAGE 1: Generate synthetic training data
# ============================================================
DATA_FILE = f"{LOCAL_ROOT}/data/synthetic_training_data.jsonl"

if os.path.exists(DATA_FILE):
    lines = len(open(DATA_FILE).readlines())
    print(f"✅ Stage 1 already complete — {lines} examples")
else:
    print("▶️  Generating training data from HuggingFace datasets...")
    print("    Uses GooAQ question-answer pairs (free, no API needed)")
    os.chdir(LOCAL_ROOT)
    !python training/01_generate_synthetic_data.py \
        --output_path data/synthetic_training_data.jsonl \
        --corpus_name sentence-transformers/gooaq \
        --n_queries 50000 \
        --n_negatives 4
    !cp data/synthetic_training_data.jsonl {DRIVE_ROOT}/data/
    print(f"✅ Stage 1 complete — saved to Drive")

os.chdir(LOCAL_ROOT)

---
## Stage 2: Knowledge Distillation
**Runtime:** ~3h on T4 | Trains ModernBERT from teacher soft labels.

Adjust `MODEL_SIZE` below:
- `"base"`  = 149M params, full fine-tune, batch 8, ~3h
- `"large"` = 395M params, LoRA fine-tune, batch 4, ~3h

> ⏱️ If session drops mid-stage, re-run. It skips if output already exists.

In [ ]:
# ============================================================
# STAGE 2: Knowledge Distillation
# ============================================================
MODEL_SIZE = "base"   # Change to "large" for ModernBERT-large
MODEL_NAME = f"answerdotai/ModernBERT-{MODEL_SIZE}"
KD_OUTPUT  = f"{LOCAL_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-en"
DRIVE_KD   = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-en"

os.chdir(LOCAL_ROOT)

if os.path.exists(KD_OUTPUT):
    print(f"✅ Stage 2 ({MODEL_SIZE}) already complete")
else:
    if not os.path.exists(f"{LOCAL_ROOT}/data/synthetic_training_data.jsonl"):
        print("❌ Run Stage 1 first (no training data found)")
    else:
        lora_flag = "--use_lora" if MODEL_SIZE == "large" else ""
        bs = 8 if MODEL_SIZE == "base" else 4
        print(f"▶️  KD: {MODEL_NAME} | LoRA: {'yes' if lora_flag else 'no'} | batch: {bs}")

        !python training/02_knowledge_distillation.py \
            --model_name {MODEL_NAME} \
            --data_path data/synthetic_training_data.jsonl \
            --output_dir {KD_OUTPUT} \
            --batch_size {bs} \
            --num_epochs 3 \
            --learning_rate 2e-5 \
            {lora_flag} \
            --lora_r 16

        if os.path.exists(KD_OUTPUT):
            os.makedirs(os.path.dirname(DRIVE_KD), exist_ok=True)
            !cp -r {KD_OUTPUT} {DRIVE_KD}
            print(f"✅ Stage 2 complete — saved to Drive")
        else:
            print("⚠️  Output not found. Check logs above.")

os.chdir(LOCAL_ROOT)

---
## Stage 2b: Multilingual Distillation (optional)
**Runtime:** ~2h | Run for cross-lingual performance. Skip if English-only is fine.

In [ ]:
# ============================================================
# STAGE 2b: Multilingual KD (optional)
# ============================================================
KD_MULTI_OUT = f"{LOCAL_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-multilingual"
DRIVE_KD_MULTI = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-multilingual"

os.chdir(LOCAL_ROOT)

if os.path.exists(KD_MULTI_OUT):
    print(f"✅ Stage 2b already complete")
else:
    print("ℹ️  Skipping multilingual KD (requires multilingual corpus)")
    print("   Uncomment the code below and set `corpus_name` to enable.")
    # !python training/02_knowledge_distillation.py \
    #     --model_name {MODEL_NAME} \
    #     --data_path data/synthetic_training_data.jsonl \
    #     --output_dir {KD_MULTI_OUT} \
    #     --batch_size 8 --num_epochs 2
    # !cp -r {KD_MULTI_OUT} {DRIVE_KD_MULTI}

os.chdir(LOCAL_ROOT)

---
## Stage 3: GRPO Reinforcement Learning
**Runtime:** ~1-2h | RL prompt warmup + fine-grained scoring.

Requires Stage 2 checkpoint as starting point.

In [ ]:
# ============================================================
# STAGE 3: GRPO RL Fine-Tuning
# ============================================================
RL_INPUT  = KD_OUTPUT
RL_OUTPUT = f"{LOCAL_ROOT}/models/flashrank-pro-{MODEL_SIZE}-rl"
DRIVE_RL  = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-rl"

os.chdir(LOCAL_ROOT)

if os.path.exists(RL_OUTPUT):
    print(f"✅ Stage 3 already complete")
else:
    if not os.path.exists(RL_INPUT):
        print(f"❌ {RL_INPUT} not found. Run Stage 2 first.")
    else:
        print(f"▶️  GRPO RL on {MODEL_SIZE} | K=8 samples | 1 epoch")
        !python training/03_grpo_rl.py \
            --model_path {RL_INPUT} \
            --data_path data/synthetic_training_data.jsonl \
            --output_dir {RL_OUTPUT} \
            --batch_size 4 \
            --k_samples 8 \
            --num_epochs 1

        if os.path.exists(RL_OUTPUT):
            os.makedirs(os.path.dirname(DRIVE_RL), exist_ok=True)
            !cp -r {RL_OUTPUT} {DRIVE_RL}
            print(f"✅ Stage 3 complete — saved to Drive")

os.chdir(LOCAL_ROOT)

---
## Stage 4: SLERP Merge
**Runtime:** ~5min | Merges KD + RL checkpoints into final model.

In [ ]:
import os, json, shutil
from pathlib import Path

os.chdir(LOCAL_ROOT)

MERGED_OUTPUT = f"{LOCAL_ROOT}/models/flashrank-pro-{MODEL_SIZE}-merged"
DRIVE_MERGED  = f"{DRIVE_ROOT}/models/flashrank-pro-{MODEL_SIZE}-merged"
KD_PATH      = f"{LOCAL_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-en"
RL_PATH      = f"{LOCAL_ROOT}/models/flashrank-pro-{MODEL_SIZE}-rl"
KD_MULTI_PATH = f"{LOCAL_ROOT}/models/flashrank-pro-{MODEL_SIZE}-kd-multilingual"

if os.path.exists(MERGED_OUTPUT):
    print(f"✅ Stage 4 already complete")
else:
    available = []
    for ckpt in [KD_PATH, KD_MULTI_PATH, RL_PATH]:
        if os.path.exists(ckpt):
            available.append(ckpt)

    if len(available) < 2:
        print(f"⚠️  Need ≥2 checkpoints, found {len(available)}. Copying single...")
        if available:
            shutil.copytree(available[0], MERGED_OUTPUT)
    else:
        weights = [1.0 / len(available)] * len(available)
        slerp_cfg = {"checkpoints": available, "weights": weights}
        os.makedirs("configs", exist_ok=True)
        with open("configs/slerp_config.json", "w") as f:
            json.dump(slerp_cfg, f)
        print(f"▶️  Merging {len(available)} checkpoints: {[Path(c).stem for c in available]}")
        !python training/04_slerp_merge.py \
            --config_path configs/slerp_config.json \
            --output_path {MERGED_OUTPUT}

    if os.path.exists(MERGED_OUTPUT):
        os.makedirs(os.path.dirname(DRIVE_MERGED), exist_ok=True)
        !cp -r {MERGED_OUTPUT} {DRIVE_MERGED}
        print(f"✅ Stage 4 complete — final model saved")

os.chdir(LOCAL_ROOT)


---
## Quick sanity check
Runs the merged model on a test query to verify it ranks sensibly.

In [ ]:
# ============================================================
# QUICK SANITY CHECK
# ============================================================
os.chdir(LOCAL_ROOT)

if os.path.exists(MERGED_OUTPUT):
    print("▶️  Loading model and running test query...")
    from flashrank_pro import Reranker
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"
    reranker = Reranker(MERGED_OUTPUT, device=device)

    query = "how to train a neural network"
    docs = [
        "Training neural networks requires backpropagation through the computational graph.",
        "Python is a high-level programming language created by Guido van Rossum.",
        "The ancient Roman Empire spanned three continents and lasted over 500 years.",
        "Gradient descent optimizes loss functions by iteratively updating parameters.",
        "Neural networks learn by adjusting weights based on error gradients.",
    ]

    results = reranker.rerank(query, docs)
    print(f"\nQuery: {query}\n")
    for i, r in enumerate(results, 1):
        print(f"  {i}. [{r['score']:.4f}] {r['text'][:80]}...")

    top_text = results[0]["text"].lower()
    if any(t in top_text for t in ["neural", "gradient", "backpropagation", "weights"]):
        print(f"\n✅ ML doc ranked first — model is working")
    else:
        print("\n⚠️  ML doc not first — may need more training")
else:
    print("⚠️  No merged model. Run Stages 1-4 first.")

os.chdir(LOCAL_ROOT)

---
## Deploy to HuggingFace
Requires `HF_TOKEN` in the 🔑 Secrets panel.
Get one at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
# ============================================================
# DEPLOY TO HUGGINGFACE HUB
# ============================================================
os.chdir(LOCAL_ROOT)

if not os.path.exists(MERGED_OUTPUT):
    print("⚠️  Run Stages 1-4 first")
else:
    hf_token = None
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        pass

    if not hf_token:
        print("⚠️  HF_TOKEN not found in 🔑 Secrets")
    else:
        repo_id = f"eulogik/flashrank-pro-{MODEL_SIZE}"
        print(f"▶️  Deploying to HuggingFace: {repo_id}")
        !python scripts/deploy_to_huggingface.py \
            --model_path {MERGED_OUTPUT} \
            --repo_id {repo_id}
        print(f"✅ https://huggingface.co/{repo_id}")

os.chdir(LOCAL_ROOT)

---
## Status Summary

All data and models persist in `MyDrive/flashrank-pro/` on Google Drive.
If your session disconnects, **Runtime → Run all** resumes automatically.

| Stage | Status | Where |
|-------|--------|-------|
| 1. Data | Check cell output ↑ | Drive/flashrank-pro/data/ |
| 2. KD | Check cell output ↑ | Drive/flashrank-pro/models/*-kd-en/ |
| 2b. Multilingual | Check cell output ↑ | Drive/flashrank-pro/models/*-kd-multilingual/ |
| 3. RL | Check cell output ↑ | Drive/flashrank-pro/models/*-rl/ |
| 4. Merge | Check cell output ↑ | Drive/flashrank-pro/models/*-merged/ |

---
**Built by [Eulogik](https://eulogik.com)** · Apache 2.0